# 09 — Rollback applied batch

This notebook prepares and executes a **rollback** for rows that were actually moved in notebook 08.

Safety defaults:
- dry-run enabled
- rollback candidates filtered to operations with `apply_status == "moved"`
- batch size limited
- full rollback log written to `data/outputs/`
- no deletes


In [1]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from src.executor import (
    ApplyConfig,
    build_rollback_runtime_manifest,
    normalize_rollback_manifest,
    rollback_manifest_apply,
    rollback_summary,
    save_rollback_log,
)


In [2]:
def latest_parquet(prefix: str) -> Path:
    candidates = sorted(OUTPUT_DIR.glob(f'{prefix}_*.parquet'))
    if not candidates:
        raise FileNotFoundError(f'No parquet found for prefix: {prefix}')
    return candidates[-1]

ROLLBACK_MANIFEST_PATH = latest_parquet('rollback_manifest')
APPLY_LOG_PATH = latest_parquet('apply_log')

DRY_RUN = True
BATCH_SIZE = 10
OVERWRITE_EXISTING = False
CREATE_TARGET_PARENTS = True

SOURCE_BASE_PATH = None
TARGET_BASE_PATH = None

print('Rollback manifest:', ROLLBACK_MANIFEST_PATH)
print('Apply log:', APPLY_LOG_PATH)
print('DRY_RUN =', DRY_RUN)
print('BATCH_SIZE =', BATCH_SIZE)


Rollback manifest: C:\00_Developement\sch-file-organizer\data\outputs\rollback_manifest_20260307_102025.parquet
Apply log: C:\00_Developement\sch-file-organizer\data\outputs\apply_log_20260307_102045.parquet
DRY_RUN = True
BATCH_SIZE = 10


In [3]:
rollback_manifest = pd.read_parquet(ROLLBACK_MANIFEST_PATH)
rollback_manifest = normalize_rollback_manifest(rollback_manifest)
apply_log = pd.read_parquet(APPLY_LOG_PATH)
runtime_manifest = build_rollback_runtime_manifest(rollback_manifest, apply_log=apply_log)

print('Rollback manifest rows:', len(rollback_manifest))
print('Apply log rows:', len(apply_log))
print('Rollback runtime rows:', len(runtime_manifest))

def _show(df, cols, n=20):
    safe_cols = [c for c in cols if c in df.columns]
    display(df[safe_cols].head(n))

_show(apply_log, ['execution_operation_id', 'apply_mode', 'apply_status', 'apply_reason'])
_show(runtime_manifest, [
    'rollback_operation_id',
    'execution_source_relative_path',
    'execution_target_relative_path',
    'execution_source_full_path',
    'execution_target_full_path',
    'execution_block_reason',
])


Rollback manifest rows: 0
Apply log rows: 0
Rollback runtime rows: 0


,execution_operation_id,apply_mode,apply_status,apply_reason


,rollback_operation_id,execution_source_relative_path,execution_target_relative_path,execution_source_full_path,execution_target_full_path,execution_block_reason


## Important

If `Rollback runtime rows` is zero, that usually means your latest apply log was a dry-run only and nothing was actually moved. That is normal.

Only operations with `apply_status == "moved"` are eligible for rollback.


In [4]:
config = ApplyConfig(
    dry_run=DRY_RUN,
    batch_size=BATCH_SIZE,
    create_target_parents=CREATE_TARGET_PARENTS,
    overwrite_existing=OVERWRITE_EXISTING,
    source_base_path=SOURCE_BASE_PATH,
    target_base_path=TARGET_BASE_PATH,
)

rollback_log = rollback_manifest_apply(rollback_manifest, apply_log=apply_log, config=config)
rollback_log


,rollback_operation_id,rollback_mode,rollback_status,rollback_reason,rollback_timestamp_utc,rollback_source_relative_path,rollback_target_relative_path,rollback_source_path,rollback_target_path


In [5]:
summary = rollback_summary(rollback_log)
summary


{'rows': 0, 'dry_run_ready': 0, 'moved_back': 0, 'blocked': 0, 'error': 0}

In [6]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path, parquet_path, jsonl_path = save_rollback_log(rollback_log, OUTPUT_DIR, ts)
print(csv_path)
print(parquet_path)
print(jsonl_path)


C:\00_Developement\sch-file-organizer\data\outputs\rollback_log_20260307_103459.csv
C:\00_Developement\sch-file-organizer\data\outputs\rollback_log_20260307_103459.parquet
C:\00_Developement\sch-file-organizer\data\outputs\rollback_log_20260307_103459.jsonl
